# Demand Forecasting — XGBoost
Monthly sales forecasting using engineered time-series features.
Train on 2014–2016, evaluate on 2017, forecast Jan–Mar 2018.

In [26]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.data_loader import load_data, clean_data
from src.features import build_features
from src.model import train_model, predict, save_model, FEATURE_COLS

TEMPLATE = "plotly_white"

df_raw = load_data("../data/superstore.csv")
df = clean_data(df_raw)
print("Data loaded:", df.shape)



Shape: (9994, 21)

Column dtypes:
row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               int64
region                   object
product_id               object
category                 object
sub_category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object

Null counts:
row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0


## Step 1 — Aggregate to Monthly Level

In [27]:
monthly = (
    df.groupby(["year", "month"])
    .agg(
        sales=("sales", "sum"),
        orders=("order_id", "nunique"),
        quantity=("quantity", "sum"),
        profit=("profit", "sum")
    )
    .reset_index()
)

monthly["date"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))
monthly = monthly.sort_values("date").reset_index(drop=True)
monthly = monthly.drop(columns=["year", "month"])

print(monthly.shape)
print(monthly.head())



(48, 5)
       sales  orders  quantity     profit       date
0  14236.895      32       284  2450.1907 2014-01-01
1   4519.892      28       159   862.3084 2014-02-01
2  55691.009      71       585   498.7299 2014-03-01
3  28295.345      66       536  3488.8352 2014-04-01
4  23648.287      69       466  2738.7096 2014-05-01


## Step 2 — Feature Engineering

In [28]:
featured = build_features(monthly)
print(featured.shape)
print(featured.head())



Feature matrix shape: (42, 15)
Date range: 2014-07-01 00:00:00 to 2017-12-01 00:00:00
Features: ['month', 'quarter', 'year', 'month_sin', 'month_cos', 'lag_1', 'lag_3', 'rolling_mean_3', 'rolling_mean_6', 'is_q4']
(42, 15)
        sales  orders  quantity     profit       date  month  quarter  year  \
0  33946.3930      65       550  -841.4826 2014-07-01      7        3  2014   
1  27909.4685      72       609  5318.1050 2014-08-01      8        3  2014   
2  81777.3508     130      1000  8328.0994 2014-09-01      9        3  2014   
3  31453.3930      78       573  3448.2573 2014-10-01     10        4  2014   
4  78628.7167     151      1219  9292.1269 2014-11-01     11        4  2014   

   month_sin     month_cos       lag_1       lag_3  rolling_mean_3  \
0  -0.500000 -8.660254e-01  34595.1276  28295.3450    28846.253200   
1  -0.866025 -5.000000e-01  33946.3930  23648.2870    30729.935867   
2  -1.000000 -1.836970e-16  27909.4685  34595.1276    32150.329700   
3  -0.866025  5.000000

## Step 3 — Train / Test Split (temporal)
Training: 2014–2016 | Test: 2017

In [29]:
train = featured[featured["date"] < "2017-01-01"]
test = featured[featured["date"] >= "2017-01-01"]

X_train = train[FEATURE_COLS]
y_train = train["sales"]

X_test = test[FEATURE_COLS]
y_test = test["sales"]

print(f"Train size: {len(train)} months")
print(f"Test size:  {len(test)} months")

Train size: 30 months
Test size:  12 months


## Step 4 — Train Model & Evaluate


In [30]:
model = train_model(X_train, y_train)

preds = predict(model, X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds) ** 0.5
r2   = r2_score(y_test, preds)

print(f"MAE:  ${mae:,.0f}")
print(f"RMSE: ${rmse:,.0f}")
print(f"R²:   {r2:.4f}")

Model trained successfully.
MAE:  $14,201
RMSE: $16,039
R²:   0.6127


## Step 5 — Actual vs Predicted Chart

In [31]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test["date"], y=y_test,
    mode="lines+markers",
    name="Actual",
    line=dict(color="#1F4E79", width=2),
    marker=dict(size=6)
))

fig.add_trace(go.Scatter(
    x=test["date"], y=preds,
    mode="lines+markers",
    name="Predicted",
    line=dict(color="#4BACC6", width=2, dash="dash"),
    marker=dict(size=6)
))

fig.update_layout(
    title=f"Actual vs Predicted Monthly Sales (2017) — R²={r2:.2f}, RMSE=${rmse:,.0f}",
    xaxis_title="Month",
    yaxis_title="Total Sales ($)",
    template=TEMPLATE,
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99)
)

fig.show()

## Step 6 — Retrain on Full Dataset & Generate Forward Forecast

In [32]:
# Retrain on all available data
X_full = featured[FEATURE_COLS]
y_full = featured["sales"]

model_full = train_model(X_full, y_full)
save_model(model_full, "../data/model.pkl")

# Build the 3 forward feature rows manually
last_known = featured.tail(6)["sales"].values  # last 6 months of actuals

forecast_dates = pd.date_range(start="2018-01-01", periods=3, freq="MS")
forecast_rows = []

# We'll track a running sales history to update lags for each forward month
sales_history = list(featured["sales"].values)

for i, date in enumerate(forecast_dates):
    lag_1 = sales_history[-1]
    lag_3 = sales_history[-3]
    roll_3 = np.mean(sales_history[-3:])
    roll_6 = np.mean(sales_history[-6:])

    month = date.month
    quarter = date.quarter
    year = date.year
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)
    is_q4 = 1 if quarter == 4 else 0

    row = {
        "month": month, "quarter": quarter, "year": year,
        "month_sin": month_sin, "month_cos": month_cos,
        "lag_1": lag_1, "lag_3": lag_3,
        "rolling_mean_3": roll_3, "rolling_mean_6": roll_6,
        "is_q4": is_q4,
        "trend": len(featured) + i
    }
    forecast_rows.append(row)

    # Predict this month and add to history for the next iteration
    X_fwd = pd.DataFrame([row])[FEATURE_COLS]
    predicted_val = predict(model_full, X_fwd)[0]
    sales_history.append(predicted_val)

forecast_df = pd.DataFrame(forecast_rows)[FEATURE_COLS]
forecast_preds = predict(model_full, forecast_df)

forecasts = pd.DataFrame({
    "date": forecast_dates,
    "predicted_sales": forecast_preds.round(2),
    "lower_bound": (forecast_preds * 0.85).round(2),
    "upper_bound": (forecast_preds * 1.15).round(2)
})

print(forecasts)
forecasts.to_csv("../data/forecasts.csv", index=False)
print("Saved to data/forecasts.csv")



Model trained successfully.
Model saved to ../data/model.pkl
        date  predicted_sales   lower_bound   upper_bound
0 2018-01-01     37680.460938  32028.390625  43332.519531
1 2018-02-01     32612.880859  27720.949219  37504.820312
2 2018-03-01     52601.500000  44711.281250  60491.718750
Saved to data/forecasts.csv


## Step 7 — Forecast Visualisation

In [33]:
# Historical actuals for context (last 12 months)
historical = monthly.tail(12)

fig = go.Figure()

# Historical line
fig.add_trace(go.Scatter(
    x=historical["date"], y=historical["sales"],
    mode="lines+markers",
    name="Historical Sales",
    line=dict(color="#1F4E79", width=2),
    marker=dict(size=5)
))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecasts["date"], y=forecasts["predicted_sales"],
    mode="lines+markers",
    name="Forecast",
    line=dict(color="#4BACC6", width=2, dash="dash"),
    marker=dict(size=8, symbol="star")
))

# Confidence band
fig.add_trace(go.Scatter(
    x=pd.concat([forecasts["date"], forecasts["date"][::-1]]),
    y=pd.concat([forecasts["upper_bound"], forecasts["lower_bound"][::-1]]),
    fill="toself",
    fillcolor="rgba(75, 172, 198, 0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    name="Confidence Band",
    showlegend=True
))

fig.update_layout(
    title="Sales Forecast — Jan to Mar 2018 (with confidence band)",
    xaxis_title="Month",
    yaxis_title="Total Sales ($)",
    template=TEMPLATE,
    hovermode="x unified"
)

fig.show()